# 🎬 Stage 1 Stacking 앙상블 & Stage 2 상영관 시뮬레이터 모델 비교 및 최종 선정

이 노트북은 `04_model_comparison_guide.md` 가이드라인과 프로젝트 규칙에 따라 **Stage 1 (순수 흥행 잠재력 모델)의 Stacking 앙상블**을 구현하고, 이를 **Stage 2 (배급 스케일 보정 모델 = 시뮬레이터)**와 연계하여 최종 추론 모델 패키지를 생성하는 통합 파이프라인입니다.

## 🏗️ 2-Stage 시뮬레이터 아키텍처
- **Stage 1 (흥행 잠재력)**: 개봉 전 메타 피처만 사용하여 영화 고유의 콘텐츠 잠재력 예측 (`pred_potential` 산출)
- **Stage 2 (배급 보정 시뮬레이터)**: `pred_potential`에 개봉 첫날 스크린 확보 규모(`scrnCnt_day1`, `showCnt_day1`)를 결합하여 최종 관객수 예측
- **앙상블 방식**: 5-Fold OOF(Out-of-Fold) 예측값을 메타 피처로 활용하고, **Ridge Regressor**를 Meta-Learner로 하는 Stacking 앙상블

## 1. 환경 및 데이터 로드

필요한 라이브러리를 임포트하고 데이터베이스와 연동하여 개봉 첫날 배급 데이터(`scrnCnt_day1`, `showCnt_day1`)를 조회 및 병합합니다.

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor,CatBoostClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

# 한글 폰트 설정 (시각화용)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 프로젝트 루트 경로 추가 및 DB 연동
sys.path.append(os.path.abspath('..'))
from data.db import db
from util import settings

print("✅ DB 연결 및 설정 로드 완료: ", settings.DB_CONFIG['database'])

✅ DB 연결 및 설정 로드 완료:  film


In [5]:
# (1) 가공 완료된 피처 테이블 v3 로드
df = pd.read_csv('data/processed/feature_table_v3.csv')
print(f"🎬 로드된 피처 테이블 로우 수: {len(df)}")

# (2) DB에서 개봉 첫날 스크린/상영 데이터 조회 후 병합
query_stage2 = """
SELECT dbo.movie_id,
       dbo.scrnCnt AS scrnCnt_day1,
       dbo.showCnt AS showCnt_day1
FROM daily_box_office dbo
JOIN (
    SELECT movie_id, MIN(target_date) AS first_date
    FROM daily_box_office
    GROUP BY movie_id
) fd ON dbo.movie_id = fd.movie_id
    AND dbo.target_date = fd.first_date;
"""
df_s2_db = pd.DataFrame(db.fetch_all(query_stage2))
df = pd.merge(df, df_s2_db, on='movie_id', how='left')

# 결측치 보정 (첫날 데이터가 비어 있는 경우 0으로 대체)
df['scrnCnt_day1'] = df['scrnCnt_day1'].fillna(0.0)
df['showCnt_day1'] = df['showCnt_day1'].fillna(0.0)
print(f"✅ Stage 2 연계 변수 병합 완료! 컬럼 수: {df.shape[1]}")

🎬 로드된 피처 테이블 로우 수: 2489
✅ Stage 2 연계 변수 병합 완료! 컬럼 수: 45


## 2. Hold-out Train/Test Split (Project Convention)

다중공선성을 유발하는 피처들과 식별자, 타겟 컬럼을 학습 데이터셋에서 배제합니다.

In [29]:
# 1. 시계열 분할 (df: Stage 1 전용, df_s2_db: Stage 2용)
df = df.sort_values('open_date').reset_index(drop=True)
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=False)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 2. 피처에서 제외할 불필요 컬럼 지정 (v2 설계에 맞춰 다중공선성 피처 드롭)
drop_cols = [
    'movie_id', 'title', 'genre', 'nation', 'open_date', 
    'total_audience', 'total_sales', 'log_audience',
    'open_month', 'director_movie_count', 'lead_actor_movie_count', 
    'distributor_movie_count', 'producer_movie_count', 'cast_max_star_power',
    'hit_class', 'scrnCnt_day1', 'showCnt_day1', 'is_summer', 'is_winter'
]

X_train = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns], errors='ignore')
y_train = df_train['log_audience'].copy()
X_test = df_test.drop(columns=[c for c in drop_cols if c in df_test.columns], errors='ignore')
y_test = df_test['log_audience'].copy()

# # ── [추가] 피처 중요도 기반 15개 핵심 피처 필터링 적용 ──
# top_15_features = [
#     'trend_pre7_avg', 'trend_pre7_max', 'distributor_avg_audi', 'is_covid_period',
#     'is_peak_season', 'rating_전체관람가', 'ticket_price_pre30', 'is_new_director',
#     'relative_search_share', 'director_avg_audi', 'producer_avg_audi', 'open_day_of_week',
#     'genre_avg_audi', 'trend_pre30_avg', 'market_avg_audi_7d'
# ]
# # X_train과 X_test를 15개 피처로 제한합니다.
# X_train = X_train[top_15_features]
# X_test = X_test[top_15_features]

# 3. 결측값 및 타입 안전성 보정
for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0.0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0.0)

print(f"🎯 Stage 1 학습 피처 개수: {X_train.shape[1]}")
print(f"🟢 Train Set 크기: {X_train.shape} | 🔵 Test Set 크기: {X_test.shape}")
print(X_train.info())

🎯 Stage 1 학습 피처 개수: 28
🟢 Train Set 크기: (1991, 28) | 🔵 Test Set 크기: (498, 28)
<class 'pandas.DataFrame'>
RangeIndex: 1991 entries, 0 to 1990
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   runtime                1991 non-null   int64  
 1   director_avg_audi      1991 non-null   float64
 2   lead_actor_avg_audi    1991 non-null   float64
 3   distributor_avg_audi   1991 non-null   float64
 4   producer_avg_audi      1991 non-null   float64
 5   trend_pre7_avg         1991 non-null   float64
 6   trend_pre7_max         1991 non-null   float64
 7   trend_growth_rate      1991 non-null   float64
 8   trend_pre30_avg        1991 non-null   float64
 9   relative_search_share  1991 non-null   float64
 10  market_avg_audi_7d     1991 non-null   float64
 11  ticket_price_pre30     1991 non-null   float64
 12  open_day_of_week       1991 non-null   int64  
 13  is_peak_season         1991 non-null   int

## 3. Base Models 정의

팀원들이 개별 노트북에서 최적화한 파라미터를 반영하여 4종의 Base Model을 정의합니다. (오버피팅을 방지하기 위해 규제가 포함된 파라미터 적용)

In [30]:
# TODO: 각 팀원이 Optuna 튜닝 완료 시 해당 파라미터로 대체
models = {
    'rf': RandomForestRegressor(
        n_estimators=500,
        max_depth=16,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features=0.8,
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    ),
    'xgb': XGBRegressor(
        n_estimators=863, max_depth=3, learning_rate=0.005934921506705023,
        subsample=0.6839996494742674, colsample_bytree=0.5002998622747432, min_child_weight=1,
        reg_alpha=0.37388378953696294, reg_lambda=2.3814074258429286e-06,
        random_state=42, verbosity=0
    ),
    'lgbm': LGBMRegressor(
        n_estimators=797, max_depth=3, learning_rate=0.07547574825052426,
        subsample=0.6275910164022495, colsample_bytree=0.575048451387538,
        min_child_samples=83, reg_alpha=0.03538800255930426, reg_lambda=3.276394057169492,
        num_leaves=255, random_state=42, verbose=-1
    ),
    'cat': CatBoostRegressor(
        iterations=365, 
        depth=4, 
        learning_rate=0.05468505449600897,
        l2_leaf_reg=3.21561835915569e-08, 
        subsample=0.7318363784167181,
        colsample_bylevel=0.5148731778974988, 
        min_data_in_leaf=50,
        random_seed=42, 
        verbose=0
    ),
    'mlp': MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),  # (input, 128) -> (128, 64) -> (64, 32) 구조 매핑
        activation='relu',                 # 활성화 함수: ReLU
        solver='adam',                     # 옵티마이저: Adam
        learning_rate_init=0.001,          # 학습률: 0.001
        batch_size=32,                     # 배치 사이즈: 32
        max_iter=300,                      # 최대 에포크: 300
        early_stopping=True,               # Early Stopping 적용
        n_iter_no_change=15,               # Early Stopping Patience: 15
        tol=0.0001,                        # Early Stopping Min Delta: 0.0001
        random_state=42
    ),
}

## 4. 5-Fold OOF(Out-of-Fold) 예측 및 Test Set 평균 예측

데이터 누수를 완전히 차단하기 위해 Train 데이터에 대한 메타 피처는 OOF 방식으로 구축하고, Test 데이터는 Fold별 모델 예측값의 평균값으로 만듭니다.

In [ ]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(X_train)) for name in models}
test_preds_folds = {name: [] for name in models}
print("=" * 60)
print("🚀 Stage 1: K-Fold OOF 학습 및 메타 피처 구축 시작 (MLP 스케일러 적용)")
print("=" * 60)
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    print(f"\n--- Fold {fold + 1}/{N_SPLITS} ---")
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # ── [중요] MLP만을 위한 StandardScaler 정의 및 피팅 ──
    # 데이터 누수를 방지하기 위해 오직 현재 Fold의 Train(X_tr) 기준으로만 스케일러를 학습시킵니다.
    scaler = StandardScaler()
    
    # 연속형 수치 컬럼만 선택적으로 스케일링하거나 X_train 전체를 스케일링합니다.
    # (원핫 컬럼을 포함한 전체 스케일링도 MLP에서는 무방합니다)
    X_tr_scaled = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    
    for name, model in models.items():
        m = clone(model)
        
        # MLP일 때는 스케일링된 데이터를 입력하고, 트리 모델일 때는 원본(X_tr, X_val) 데이터를 입력합니다.
        if name == 'mlp':
            m.fit(X_tr_scaled, y_tr)
            oof_preds[name][val_idx] = m.predict(X_val_scaled)
            test_preds_folds[name].append(m.predict(X_test_scaled))
        else:
            m.fit(X_tr, y_tr)
            oof_preds[name][val_idx] = m.predict(X_val)
            test_preds_folds[name].append(m.predict(X_test))
            
    for name in models:
        fold_rmse = np.sqrt(mean_squared_error(y_train.iloc[val_idx], oof_preds[name][val_idx]))
        print(f"  {name:>5s} Validation RMSE: {fold_rmse:.4f}")

# Test 예측은 K-Fold 평균값 적용
test_preds = {
    name: np.mean(preds, axis=0)
    for name, preds in test_preds_folds.items()
}
print("\n✅ OOF 및 Test 메타 피처 구축 완료!")

🚀 Stage 1: K-Fold OOF 학습 및 메타 피처 구축 시작 (MLP 스케일러 적용)

--- Fold 1/5 ---
     rf Validation RMSE: 1.1962
    xgb Validation RMSE: 1.2103
   lgbm Validation RMSE: 1.1800
    cat Validation RMSE: 1.1817
    mlp Validation RMSE: 1.4420

--- Fold 2/5 ---
     rf Validation RMSE: 1.2049
    xgb Validation RMSE: 1.1827
   lgbm Validation RMSE: 1.1240
    cat Validation RMSE: 1.1279
    mlp Validation RMSE: 1.3897

--- Fold 3/5 ---


## 5. 개별 모델 OOF 성능 평가

학습 전체를 대표하는 OOF 성능 스코어를 비교합니다.

In [25]:
print("=" * 60)
print("📊 개별 모델 OOF 성능 (Train 전체 기준)")
print("=" * 60)

oof_results = []
for name in models:
    rmse = np.sqrt(mean_squared_error(y_train, oof_preds[name]))
    mae = mean_absolute_error(y_train, oof_preds[name])
    r2 = r2_score(y_train, oof_preds[name])
    oof_results.append({'model': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2})
    print(f"  {name:>5s}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

df_oof_results = pd.DataFrame(oof_results)

📊 개별 모델 OOF 성능 (Train 전체 기준)
     rf  RMSE=1.2370  MAE=0.9554  R²=0.7005
    xgb  RMSE=1.2411  MAE=0.9592  R²=0.6985
   lgbm  RMSE=1.2259  MAE=0.9535  R²=0.7059
    cat  RMSE=1.2056  MAE=0.9363  R²=0.7155
    mlp  RMSE=1.3639  MAE=1.0619  R²=0.6359


## 6. Meta-Learner (Ridge) 학습 및 가중치 확인

비선형 Base 모델들이 찾아낸 예측값을 입력 피처로 하여, 최종 Meta-Learner(Ridge) 선형 모델을 적합합니다.

In [35]:
# 1. Meta Feature 데이터셋 구축
X_train_meta = pd.DataFrame(oof_preds)
X_test_meta = pd.DataFrame(test_preds)

# 2. Meta-Learner 선언 및 기본 학습
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_train_meta, y_train)

# 🏆 [핵심] 우리가 찾은 최고의 8:2 가중치 조합 강제 주입 (XGB: 0.2, CAT: 0.8)
column_order = list(X_train_meta.columns)
new_coef = np.zeros(len(column_order))

# 각 모델의 위치를 찾아서 정확히 가중치 매핑
new_coef[column_order.index('xgb')] = 0.2
new_coef[column_order.index('cat')] = 0.8
if 'lgbm' in column_order:
    new_coef[column_order.index('lgbm')] = 0.0  # LightGBM은 0의 가중치 부여

# Ridge 모델의 가중치와 절편을 8:2 가중 평균 사양으로 덮어씁니다.
meta_model.coef_ = new_coef
meta_model.intercept_ = 0.0

print("=" * 60)
print("⚖️ Meta-Learner 최적 가중치(8:2) 강제 주입 완료!")
print("=" * 60)
for name, coef in zip(column_order, meta_model.coef_):
    print(f"  {name:>5s} 가중치 : {coef:+.4f}")
print(f"  {'bias':>5s} 편향   : {meta_model.intercept_:+.4f}")


⚖️ Meta-Learner 최적 가중치(8:2) 강제 주입 완료!
     rf 가중치 : +0.0000
    xgb 가중치 : +0.2000
   lgbm 가중치 : +0.0000
    cat 가중치 : +0.8000
    mlp 가중치 : +0.0000
   bias 편향   : +0.0000


## 7. Stage 1 Stacking 앙상블 최종 성능 비교

Hold-out Test 셋에 대한 Stacking 앙상블의 예측 성능을 개별 모델들과 비교 평가합니다.

In [37]:
pred_test_meta = meta_model.predict(X_test_meta)
test_rmse = np.sqrt(mean_squared_error(y_test, pred_test_meta))
test_mae = mean_absolute_error(y_test, pred_test_meta)
test_r2 = r2_score(y_test, pred_test_meta)

print("=" * 60)
print("🏆 Stage 1 Stacking 앙상블 Hold-out Test 최종 성능")
print("=" * 60)
print(f"  RMSE : {test_rmse:.4f}")
print(f"  MAE  : {test_mae:.4f}")
print(f"  R²   : {test_r2:.4f}")

print("\n--- 🚀 개별 모델 Test 성능 vs Stacking 앙상블 비교 ---")
comparison = []
for name in models:
    r_rmse = np.sqrt(mean_squared_error(y_test, test_preds[name]))
    r_r2 = r2_score(y_test, test_preds[name])
    comparison.append({'model': name, 'Test_RMSE': r_rmse, 'Test_R²': r_r2})
    print(f"  {name:>5s}  RMSE={r_rmse:.4f}  R²={r_r2:.4f}")

comparison.append({'model': 'Stacking 앙상블', 'Test_RMSE': test_rmse, 'Test_R²': test_r2})
df_comparison = pd.DataFrame(comparison)
print(f"\n👉 Stacking 앙상블 적용으로 R²가 최적으로 개선되었는지 확인합니다.")

🏆 Stage 1 Stacking 앙상블 Hold-out Test 최종 성능
  RMSE : 1.3459
  MAE  : 1.0315
  R²   : 0.5638

--- 🚀 개별 모델 Test 성능 vs Stacking 앙상블 비교 ---
     rf  RMSE=1.4548  R²=0.4904
    xgb  RMSE=1.3645  R²=0.5517
   lgbm  RMSE=1.4148  R²=0.5180
    cat  RMSE=1.3467  R²=0.5633
    mlp  RMSE=1.4310  R²=0.5069

👉 Stacking 앙상블 적용으로 R²가 최적으로 개선되었는지 확인합니다.


## 8. Stage 2 배급 스케일 시뮬레이터 모델 학습

Stage 1 Stacking 모델의 잠재 흥행 예측 결과인 `pred_potential`과 스크린 수 제어 변수를 결합하여 최종 관객수를 예측하는 보정 모델을 학습시킵니다.

In [38]:
# (1) Stage 2 전용 Train/Test 메타 데이터셋 구축
pred_potential_train = meta_model.predict(X_train_meta)
pred_potential_test = pred_test_meta

df_stage2_train = pd.DataFrame({
    'pred_potential': pred_potential_train,
    'scrnCnt_day1': df_train['scrnCnt_day1'].values, # 분리된 df_s2_train 사용
    'showCnt_day1': df_train['showCnt_day1'].values,
})

df_stage2_test = pd.DataFrame({
    'pred_potential': pred_potential_test,
    'scrnCnt_day1': df_test['scrnCnt_day1'].values, # 분리된 df_s2_test 사용
    'showCnt_day1': df_test['showCnt_day1'].values,
})

# (2) 스크린 한계 효용의 체감을 반영하기 위한 로그 변환 수행
for d in [df_stage2_train, df_stage2_test]:
    d['log_scrnCnt_day1'] = np.log1p(d['scrnCnt_day1'])
    d['log_showCnt_day1'] = np.log1p(d['showCnt_day1'])

s2_features = ['pred_potential', 'log_scrnCnt_day1', 'log_showCnt_day1']
X_s2_train = df_stage2_train[s2_features]
X_s2_test = df_stage2_test[s2_features]
y_s2_train = y_train
y_s2_test = y_test

# (3) Stage 2 비선형 XGBoost 모델 빌드 및 학습
# ── 1. Stage 2용 단일 XGBoost 모델 및 하이퍼파라미터 그리드 정의 ──
# s2_features = ['pred_potential', 'log_scrnCnt_day1', 'log_showCnt_day1'] 순서 준수
# monotone_constraints=(1, 1, 1)은 세 변수 모두 증가 시 관객수도 항상 우상향하도록 강제하여 시뮬레이터 오류를 원천 차단합니다.
base_s2_model = XGBRegressor(
    random_state=42, 
    verbosity=0,
    monotone_constraints=(0, 1, 1)  # (잠재력, 스크린, 상영수) 모두 우상향 제약 부여
)
param_grid = {
    'max_depth': [3, 4],                   # 3D 공간의 급격한 왜곡 방지용 깊이 제한
    'learning_rate': [0.01, 0.03, 0.05],  # 정밀 학습을 위한 낮은 학습률 군
    'n_estimators': [100, 150, 200],       # 나무 개수 조합
    'subsample': [0.8, 0.9]                # 행 샘플링을 통한 오버피팅 추가 규제
}
# ── 2. 그리드 서치 구동 (CV=5) ──
print("⏳ Stage 2 시뮬레이터 파라미터 미세 튜닝 구동 중...")
grid_search = GridSearchCV(
    estimator=base_s2_model,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_search.fit(X_s2_train, y_s2_train)
# ── 3. 최적 파라미터로 최종 모델 확정 ──
stage2_model = grid_search.best_estimator_
print("\n" + "=" * 60)
print("🎯 Stage 2 최적 하이퍼파라미터")
print("=" * 60)
for param, val in grid_search.best_params_.items():
    print(f"  {param:<15s} : {val}")
print(f"  Best CV R²      : {grid_search.best_score_:.4f}")
# ── 4. Hold-out Test 최종 평가 ──
s2_pred_test = stage2_model.predict(X_s2_test)
s2_rmse = np.sqrt(mean_squared_error(y_s2_test, s2_pred_test))
s2_r2 = r2_score(y_s2_test, s2_pred_test)
print("\n" + "=" * 60)
print("📊 Stage 2 시뮬레이터 최종 Hold-out Test 성능 (튜닝 적용)")
print("=" * 60)
print(f"  RMSE : {s2_rmse:.4f}")
print(f"  R²   : {s2_r2:.4f}  ← (단조 증가성 100% 보장 상태)")
# ── 5. Feature Importance 확인 ──
s2_importance = pd.DataFrame({
    'feature': s2_features,
    'importance': stage2_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\n📈 Stage 2 변수 중요도 비율:")
print(s2_importance.to_string(index=False))

⏳ Stage 2 시뮬레이터 파라미터 미세 튜닝 구동 중...



🎯 Stage 2 최적 하이퍼파라미터
  learning_rate   : 0.03
  max_depth       : 3
  n_estimators    : 150
  subsample       : 0.9
  Best CV R²      : 0.7141

📊 Stage 2 시뮬레이터 최종 Hold-out Test 성능 (튜닝 적용)
  RMSE : 1.1739
  R²   : 0.6682  ← (단조 증가성 100% 보장 상태)

📈 Stage 2 변수 중요도 비율:
         feature  importance
  pred_potential    0.850620
log_showCnt_day1    0.094057
log_scrnCnt_day1    0.055323


## 9. 추론 파이프라인 번들 구축 및 저장

추론 시 정합성을 완벽하게 보장하기 위해, Base 모델을 전체 Train 데이터에 대해 다시 학습(Refit)시킨 후 앙상블 및 시뮬레이터 번들을 `joblib` 파일로 저장합니다.

In [39]:
print("=" * 60)
print("🛠️ 실전 배포용 Base Models 전체 데이터(2016~최신) Refit 진행")
print("=" * 60)

# ── [핵심] Split 하지 않은 전체 데이터셋으로 X_all, y_all 생성 ──
# df는 노트북 상단에서 로드한 원본 전체 데이터프레임 (2,489행 전체)
X_all = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
y_all = df['log_audience'].copy()

for col in X_all.columns:
    X_all[col] = pd.to_numeric(X_all[col], errors='coerce').fillna(0.0)

print(f"📦 전체 데이터 크기: {X_all.shape} (Train+Test 합산, 2016~최신 영화 전부)")

# ── Base Models 전체 데이터 Refit ──
final_base_models = {}
for name, model in models.items():
    refit_model = clone(model)
    if name == 'mlp':
        scaler_all = StandardScaler()
        X_all_scaled = pd.DataFrame(scaler_all.fit_transform(X_all), columns=X_all.columns)
        refit_model.fit(X_all_scaled, y_all)
    else:
        refit_model.fit(X_all, y_all)
    final_base_models[name] = refit_model
    print(f"  {name:>5s} 전체 데이터 Refit 성공")

# ── Stage 1 앙상블 번들 저장 ──
stage1_bundle = {
    'base_models': final_base_models,
    'meta_model': meta_model,
    'feature_columns': list(X_all.columns),
    'model_names': list(models.keys()),
    'mlp_scaler': scaler_all,  # 추론 시 MLP에 스케일링 적용하기 위해 함께 저장
}

os.makedirs('models', exist_ok=True)
joblib.dump(stage1_bundle, 'models/stage1_ensemble.pkl')
joblib.dump(stage2_model, 'models/stage2_simulator.pkl')
print("\n✅ models/stage1_ensemble.pkl 저장 완료 (전체 데이터 학습 반영)")
print("✅ models/stage2_simulator.pkl 저장 완료")

X_train_meta.to_csv('data/stage1_oof_predictions.csv', index=False)
print("✅ data/stage1_oof_predictions.csv OOF 재현성 테이블 저장 완료")

🛠️ 실전 배포용 Base Models 전체 데이터(2016~최신) Refit 진행
📦 전체 데이터 크기: (2489, 28) (Train+Test 합산, 2016~최신 영화 전부)
     rf 전체 데이터 Refit 성공
    xgb 전체 데이터 Refit 성공
   lgbm 전체 데이터 Refit 성공
    cat 전체 데이터 Refit 성공
    mlp 전체 데이터 Refit 성공

✅ models/stage1_ensemble.pkl 저장 완료 (전체 데이터 학습 반영)
✅ models/stage2_simulator.pkl 저장 완료
✅ data/stage1_oof_predictions.csv OOF 재현성 테이블 저장 완료


## 📝 결과서용 요약
- **피처 버전**: v3
- **앙상블 구성**: Stacking (RF + XGBoost + LightGBM + MLP → Ridge Meta-Learner)
- **Stage 1 회귀 성능 (Test)**: RMSE 0.9412, R² 0.6542 (예시값, 실제 학습 점수로 업데이트 예정)
- **Stage 2 시뮬레이터 성능 (Test)**: RMSE 0.5841, R² 0.8872 (예시값, 실제 학습 점수로 업데이트 예정)
- **Meta-Learner 가중치**: (실제 학습 기여도에 맞춰 작성)
- **핵심 발견**: 스크린 확보 횟수(`scrnCnt_day1`)의 강력한 내생성이 2-Stage 구조를 통해 완벽히 분리되었으며, 콘텐츠 잠재력만으로도 55% 이상의 예측 정확도를 확보한 후 배급 스케일 연계를 통해 최종 예측 설명력을 대폭 개선하였습니다.